# Reproducing Liu, Nair et al. (2024) from dandiset 001097

Paper: Liu M, Nair A, Coria N, Linderman SW, Anderson DJ (2024). Encoding of female mating
dynamics by a hypothalamic line attractor. *Nature* 634, 901-909.
[doi:10.1038/s41586-024-07916-w](https://doi.org/10.1038/s41586-024-07916-w). Open access (CC BY,
PMC11499253); the full text, Methods and Extended Data legends were used.

Data: [DANDI:001097](https://dandiarchive.org/dandiset/001097/0.260624.0330), version
0.260624.0330, streamed from the archive.

Why this version. The paper cites the dandiset by accession number only. Of the three published
versions, 0.240814.1849 holds only the two per-mouse files; 0.260531.0005 and 0.260624.0330 hold the
same 107 files with the same asset identifiers (shown below). The most recent, 0.260624.0330, is
pinned. The two per-mouse files are byte-identical across all three versions.

Code read: [neuralcomputation/hypothalamic-dynamics](https://github.com/neuralcomputation/hypothalamic-dynamics)
at commit 28fee70, which the dandiset links. It holds demonstration notebooks that fit rSLDS models
with the `ssm` package ([lindermanlab/ssm](https://github.com/lindermanlab/ssm) at commit eb6c8aa)
to one mating session (the per-mouse file of m541), and loading, smoothing, autocorrelation and
plotting functions. It does not hold the code for the paper's figures. Where it settles a choice
the Methods leave open (smoothing, the autocorrelation half-width, which cells count as weighted)
it is followed here, and the helper module says so.

## What can and cannot be tested

The dandiset holds the imaging and annotations of the 15 receptive mice of Figures 2 and 3, the
oestrogen-injected cohort of Figure 5i-l (42 trials where the paper has 50), natural-cycle and
ovariectomy series, and male-inhibition sessions. It holds no fitted rSLDS models apart from the
per-cell weights of two mice.

The rSLDS cannot be refit here. The `ssm` package is a Cython extension installed from GitHub that
does not build in the registry's locked environment, and with the authors' settings (3 states, 2 dimensions, 150 iterations of Laplace EM) one fit of
a 10,000-frame session takes about 7.5 minutes (10 iterations took 30 s), so the 90 or so fits the claims need would take hours. (These timings, and the stand-in below,
were measured while preparing this attempt and are not run in this notebook.) A single linear dynamical system,
tried as a stand-in while preparing this attempt, fitted rotations with two equal time constants in
14 of the 15 receptive sessions, so it cannot separate an integration dimension from the others
and is not a test of the claims. Claims that need new rSLDS fits are blocked.

| Claim | Paper | Here |
|---|---|---|
| `fig1b-accepting-escalates` | Accepting behaviour rises over mating; resistance rises then falls | Tested |
| `fig1c-responsive-dominates` | 83.9% of female mating time is male-responsive | Tested |
| `fig1d-male-drives-mating` | Males spend 11.3 times more time in self-initiated mating behaviour | Tested |
| `fig1e-interbout-interval` | Mean interbout interval 13.7 s | Tested |
| `fig1g-social-in-bouts-and-intervals` | Social behaviour for 62% of bouts and 23% of intervals | Tested |
| `fig2c-few-tuned-cells` | About 2 to 13% of cells tuned to a behaviour | Tested |
| `fig2e-glm-behaviour-explains-little` | Behaviour GLMs explain 8 to 15% of variance | Tested |
| `fig2f-glm-coupling` | Adding the other cells raises it to 46% | Tested |
| `fig2i-persistent-cells` | 30% of cells have ACHW above 25 s; mean 20 s | Tested |
| `ed3b-pencil-cup` | ACHW shorter with the male behind a pencil cup | Blocked: the period is not marked |
| `fig3a-rslds-fit` | rSLDS explains 64% of variance | Blocked: needs rSLDS fits |
| `fig3c-integration-dimension` | One dimension with a 111-s time constant against 25 s | Blocked: needs rSLDS fits |
| `ed3s-integrator-model` | Male behaviour drives an integrator with a time constant above 200 s | Blocked: needs rSLDS fits |
| `ed4b-single-cell-ramps` | 56% of integration cells ramp | Blocked: needs rSLDS fits |
| `fig3k-perturbation-relaxation` | Activity relaxes back to the attractor after inhibition | Blocked: the sessions are not in the dandiset |
| `fig4b-persistence-during-intervals` | Integration dimension as high in intervals as in bouts | Blocked: needs rSLDS fits |
| `fig4d-persistence-male-inhibition` | It stays high while the male is inhibited | Blocked: needs rSLDS fits |
| `fig4h-peak-before-ejaculation` | It peaks before ejaculation | Blocked: needs rSLDS fits |
| `fig5c-attractor-receptive-days` | Line attractor on receptive days only | Blocked: needs rSLDS fits |
| `ed6e-projection-unreceptive-days` | The receptive-day dimension stays low on unreceptive days | Tested with the authors' weights, 2 mice |
| `ed7-achw-across-cycle` | Weighted cells are less persistent on unreceptive days | Tested with the authors' weights, 2 mice |
| `ed9j-ovx-hormone-priming` | The attractor returns with hormone priming | Blocked: needs rSLDS fits |
| `fig5k-receptivity-correlation` | Integration dimension tracks receptivity (r2 0.62) | Blocked: needs rSLDS fits |
| `ed10c-population-mean-uncorrelated` | The population mean does not track receptivity | Tested |

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vmhvl

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Versions of the dandiset

The paper cites the dandiset by accession number only, with no version. Three versions are
published. The comparison below lists, for each, the files and their asset identifiers.

In [ ]:
versions = ["0.240814.1849", "0.260531.0005", "0.260624.0330"]
tables = {v: vmhvl.session_table(v) for v in versions}
for v, t in tables.items():
    print(f"{v}: {len(t)} files, {t.subject.nunique()} subject labels, {t['size'].sum() / 1e6:.0f} MB")
same = tables["0.260531.0005"][["path", "asset_id"]].equals(tables["0.260624.0330"][["path", "asset_id"]])
print("0.260531.0005 and 0.260624.0330 hold the same files with the same asset ids:", same)
old = tables["0.240814.1849"]
print("0.240814.1849 holds:", old.path.tolist())
print("  the same paths in 0.260624.0330 have the same asset ids:",
      old.set_index("path").asset_id.equals(tables["0.260624.0330"].set_index("path").asset_id.loc[old.path]))
print("  and the same sizes:",
      old.set_index("path")["size"].equals(tables["0.260624.0330"].set_index("path")["size"].loc[old.path]))
for p in old.path:
    a, b = vmhvl.digest(p, "0.240814.1849"), vmhvl.digest(p, "0.260624.0330")
    print(f"  {p}: SHA-256 {a[:12]}... in 0.240814.1849 and {b[:12]}... in 0.260624.0330; identical {a == b}")

## Load the data and check it against its description

Every file's annotations are read, and the traces of the sessions the analysis uses: the 15
receptive sessions, the oestrogen-injected cohort, the natural-cycle days of the two mice with
deposited rSLDS weights, and a few sessions needed to check duplicates.

In [ ]:
files = tables[vmhvl.VERSION]
sessions_t = files[files.session.notna()].copy()
annotations = vmhvl.load_many(sessions_t.path.tolist(), neural=False)
sessions_t["n_frames"] = [annotations[p].n_frames for p in sessions_t.path]
sessions_t["n_cells"] = [annotations[p].n_cells for p in sessions_t.path]
sessions_t["kind"] = [annotations[p].kind for p in sessions_t.path]
sessions_t["female_annotated"] = [annotations[p].has_female() for p in sessions_t.path]
sessions_t["receptive"] = [annotations[p].receptive() for p in sessions_t.path]

receptive_paths = sessions_t[sessions_t.experiment == "receptive"].path.tolist()
oestrogen_paths = sessions_t[sessions_t.experiment == "oestrogen-injected (7 days)"].path.tolist()
cycle_paths = sessions_t[(sessions_t.experiment == "natural cycle (3 days)")
                         & sessions_t.subject.isin(["m541", "543"])].path.tolist()
duplicate_pairs = [
    ("sub-m764_ses-WT_3d_mating_d1_ecephys+behavior.nwb", "sub-m764_ses-ovx_primed_receptive_mating_ecephys+behavior.nwb"),
    ("sub-m776_ses-ovx_primed_receptive_mating_ecephys+behavior.nwb", "sub-m776_ses-wireless_opto_ecephys+behavior.nwb"),
    ("sub-m551_ses-OVX_3d_mating_d3_ecephys+behavior.nwb", "sub-m551_ses-ovx_primed_receptive_mating_ecephys+behavior.nwb"),
]
neural_paths = sorted(set(receptive_paths + oestrogen_paths + cycle_paths + [p for pair in duplicate_pairs for p in pair]))
data = vmhvl.load_many(neural_paths, neural=True)
weights = {s: vmhvl.load_weights(s) for s in ("m541", "m542")}
print(f"annotations read from {len(annotations)} files, traces from {len(data)} files")

In [ ]:
print("Files by experiment (sessions, distinct subject labels, cells per session, frames per session):")
summary = sessions_t.groupby("experiment").agg(
    sessions=("path", "size"), subjects=("subject", "nunique"),
    cells_min=("n_cells", "min"), cells_max=("n_cells", "max"),
    frames_min=("n_frames", "min"), frames_max=("n_frames", "max"),
    female_annotated=("female_annotated", "sum"), receptive=("receptive", "sum"))
print(summary.to_string())
print()
print("Subjects per experiment:")
for exp, g in sessions_t.groupby("experiment"):
    print(f"  {exp}: {', '.join(sorted(g.subject.unique()))}")
print()
print("Stored traces:")
for kind, g in sessions_t.groupby("kind"):
    s = annotations[g.path.iloc[0]]
    print(f"  {kind}: {len(g)} files, unit '{s.unit}', description '{s.description[:60]}...'")
for kind in ("Fluorescence", "DfOverF"):
    ps = [p for p in neural_paths if data[p].kind == kind]
    allv = np.concatenate([data[p].traces.ravel() for p in ps])
    means = np.concatenate([data[p].traces.mean(0) for p in ps])
    sds = np.concatenate([data[p].traces.std(0) for p in ps])
    print(f"  {kind} (loaded {len(ps)} files): values {allv.min():.1f} to {allv.max():.1f}; "
          f"per-cell mean {means.min():.2f} to {means.max():.2f}; per-cell SD {sds.min():.2f} to {sds.max():.2f}; "
          f"NaN {np.isnan(allv).sum()}")
dts = np.concatenate([np.diff(data[p].timestamps) for p in neural_paths])
starts = [data[p].timestamps[0] for p in neural_paths]
print(f"  timestamps: first {min(starts):.3f} to {max(starts):.3f} s; steps {dts.min():.4f} to {dts.max():.4f} s "
      f"(rate {1 / np.median(dts):.2f} Hz)")
print()
b_all = pd.concat([a.behaviour.assign(path=p) for p, a in annotations.items()])
on_grid = np.abs(b_all[["start", "stop"]].to_numpy() * 30 - np.round(b_all[["start", "stop"]].to_numpy() * 30))
print(f"Annotations: {len(b_all)} intervals; stop before start in {(b_all.stop < b_all.start).sum()}; "
      f"zero-length {(b_all.stop == b_all.start).sum()}; times off the 30-Hz grid by more than 1 ms: {(on_grid > 0.03).sum()}")
ends = pd.Series({p: annotations[p].behaviour.stop.max() - annotations[p].n_frames / vmhvl.RATE for p in annotations})
print(f"  last annotation minus end of imaging: {ends.min():.1f} to {ends.max():.1f} s")
print(f"  sessions whose imaging runs more than 60 s past the last annotation: "
      f"{', '.join(p.split('_ecephys')[0] for p in ends[ends < -60].index)}")
entry = pd.Series({p: annotations[p].interaction()[0] for p in annotations})
print(f"  male entry (into_male_cage) at {entry.min():.0f} to {entry.max():.0f} s after the start of imaging")
print("  labels, Ch1 (male):", sorted(b_all[b_all.channel == "Ch1"].behavior.unique()))
print("  labels, Ch2 (female):", sorted(b_all[b_all.channel == "Ch2"].behavior.unique()))
overlap = []
for p, a in annotations.items():
    f = a.behaviour[(a.behaviour.channel == "Ch2") & (a.behaviour.behavior != "into_male_cage")].sort_values("start")
    overlap.append(np.clip(f.stop.to_numpy()[:-1] - f.start.to_numpy()[1:], 0, None).sum())
print(f"  overlap between consecutive female annotations, summed over all sessions: {sum(overlap):.1f} s")
print()
print("Descriptive metadata, distinct values over the session files:")
meta = pd.DataFrame([a.meta for a in annotations.values()])
for col in meta.columns:
    print(f"  {col}: {sorted(meta[col].astype(str).unique())}")

Checks against the files' own descriptions and the paper:

- Counts. 15 receptive mice, as in Figures 2 and 3. The oestrogen-injected cohort has the
  paper's 6 mice but 42 trials, where Figure 5k has 50. The natural-cycle series has 7 mice where
  the paper has 4, and the male-inhibition experiment has 8 where Figure 4d has 4; which ones the
  paper used is not recorded. Female behaviour is annotated in 70 sessions (69 distinct trials
  from 23 mice, below), where Figure 1 has 74 trials from 28 mice.
- Rates. The traces are at 10 Hz, as the files and Methods say; every timestamp step is 0.1 s.
  Every annotation time lies on the 30-Hz grid the descriptions give.
- Units of the raw traces. The 33 `Fluorescence` series are described as raw fluorescence in
  lumens. Their values run from about -5,500 to 8,100, with negative values and per-cell means
  from 2 to 1,400, which is not a light flux; they are consistent with CNMF-E traces in arbitrary
  units. They are only z-scored here, so the unit does not matter to the analysis.
- The z-scored traces. The 72 `DfOverF` series are described as z-scored, but per-cell means run
  from -1.6 to 2.4 and SDs from 0.1 to 1.8. This is consistent with the Methods, which z-score
  the concatenated days of a longitudinal series and then split them, though the files do not say
  so. They are used as stored where the analysis compares days, and z-scored again where it does
  not.
- Imaging metadata. Every session file says the device is a two-photon microscope with 920-nm
  excitation; the Methods describe a one-photon miniscope (Inscopix nVista). Experimenter, lab and
  institution are placeholders, the session start time is the date the file was written, the
  image masks are 1 by 1 pixel so there are no cell footprints, and the file names say `ecephys`
  for imaging data. The per-mouse files describe their session as a resident-intruder assay.
- Annotations. Male entry falls 123 to 233 s after the start of imaging, consistent with the
  2-min baseline of the Methods. In 17 sessions imaging runs more than a minute past the last
  annotation, by up to 7 minutes; what that period holds is not described. The labels `other`
  and `eat` are not described in the Methods, the male channel of two m551 sessions holds `sniff`,
  and one of them `grooming`, the female channel of one session holds `intromission`, and the male-inhibition
  epochs are labelled `laser_on` in four sessions and `mating_inhibition` in the other four.
  Female annotations barely overlap (23 s summed over all sessions), so their durations can be
  added.
- Subjects. One mouse is labelled `543` in its natural-cycle files and `m543` in its receptive
  session.

### Duplicated recordings and the per-mouse files

Some sessions have the same number of frames and cells as sessions filed under another
experiment. The cell below compares their traces, cell by cell after z-scoring, and their
annotations. It then matches the traces of the two per-mouse files, which carry the authors'
rSLDS weights, to the natural-cycle sessions.

In [ ]:
def match(a, b):
    """Median over cells of the best correlation of each cell of a with any cell of b."""
    n = min(len(a), len(b))
    za, zb = stats.zscore(a[:n], axis=0), stats.zscore(b[:n], axis=0)
    return np.median(np.max(za.T @ zb / n, axis=1))

for p, q in duplicate_pairs:
    a, b = data[p], data[q]
    same_shape = a.traces.shape == b.traces.shape
    r = np.median([np.corrcoef(a.traces[:, i], b.traces[:, i])[0, 1] for i in range(a.n_cells)]) if same_shape else np.nan
    ca = a.behaviour[a.behaviour.channel == "Ch1"].behavior.value_counts().to_dict()
    cb = b.behaviour[b.behaviour.channel == "Ch1"].behavior.value_counts().to_dict()
    print(f"{p.split('_ecephys')[0]} ({a.kind}) and {q.split('_ecephys')[0]} ({b.kind}): shape {a.traces.shape} / {b.traces.shape}; "
          f"median correlation of cell i with cell i {r:.4f}; male annotations identical {ca == cb}")
print()
for s, w in weights.items():
    print(f"per-mouse file sub-{s}: subject_id '{w['subject_id']}', traces {w['traces'].shape}, weight columns {w['columns']}, "
          f"session_description '{w['session_description']}', comments '{w['comments']}'")
    for p in cycle_paths:
        d = data[p]
        if d.traces.shape[1] == w["traces"].shape[1]:
            print(f"    {p.split('_ecephys')[0]}: {d.traces.shape}, best-match correlation {match(w['traces'], d.traces):.2f}")
        else:
            print(f"    {p.split('_ecephys')[0]}: {d.traces.shape}, cell counts differ")
m542_days = sessions_t[(sessions_t.subject == "m542") & sessions_t.experiment.str.startswith("natural")]
print(f"    the natural-cycle sessions of m542 have {sorted(set(m542_days.n_cells))} cells")

### Which natural-cycle day is the receptive one

The files do not say which day of the natural-cycle series was receptive, and the female's
behaviour is not annotated on those days, so the paper's definition (the female shows accepting
behaviour) cannot be applied. Each mouse's receptive session is also deposited separately
(`wt_proestrus_receptive_mating`). The day whose male annotations and traces match that session
is taken as the receptive day.

In [ ]:
def male_onsets(s):
    b = s.behaviour
    return np.sort(b[(b.channel == "Ch1") & b.behavior.isin(vmhvl.MALE_SELF)].start.to_numpy())

receptive_day = {}
cycle = sessions_t[sessions_t.experiment == "natural cycle (3 days)"]
for subject, g in cycle.groupby("subject"):
    rec = sessions_t[(sessions_t.experiment == "receptive") & (sessions_t.subject.str.lstrip("m") == subject.lstrip("m"))]
    if rec.empty:
        print(f"{subject}: no receptive session deposited")
        continue
    r = annotations[rec.path.iloc[0]]
    for _, row in g.sort_values("day").iterrows():
        d = annotations[row.path]
        a, b = male_onsets(d), male_onsets(r)
        same_n = len(a) == len(b)
        offset = np.median(b - a) if same_n else np.nan
        spread = np.ptp(b - a) if same_n else np.nan
        line = f"{subject} day {row.day}: {len(a)} male annotations against {len(b)}"
        if same_n:
            line += f", offset {offset:.2f} s (range {spread:.2f} s)"
        if row.path in data and rec.path.iloc[0] in data:
            line += f", trace best-match correlation {match(data[row.path].traces, data[rec.path.iloc[0]].traces):.2f}"
        print(line)
        if same_n and spread < 0.5:
            receptive_day[subject] = row.day
print("receptive day by matching:", receptive_day)

### Do the traces and the annotations share a clock?

For every session whose traces are used, a decoder predicts male contact (male sniff, mount or
intromission) from 1-s bins of population activity during the interaction, with the labels
shifted against the traces. If both share a clock, accuracy should be highest near zero shift,
or a second or two later given the calcium indicator, and fall off at shifts of half a minute or
more.

In [ ]:
rng_clock = np.random.default_rng(1097_00)
shifts = [-60, -30, -10, -3, 0, 3, 10, 30, 60]
clock = []
for p in receptive_paths + oestrogen_paths + cycle_paths:
    s = data[p]
    t0, t1 = s.interaction()
    i0, i1 = int(round(t0 * vmhvl.RATE)), min(int(round(t1 * vmhvl.RATE)), s.n_frames)
    y = vmhvl.bin_frames(vmhvl.zscore(s.traces)[i0:i1], 10)
    lab = vmhvl.bin_frames(vmhvl.raster(s.behaviour, vmhvl.MALE_SELF, s.n_frames, "Ch1")[i0:i1].astype(float), 10) >= 0.5
    acc = vmhvl.shifted_decoding(y, lab, shifts, rng_clock)
    clock.append({"session": p.split("_ecephys")[0], "positives": int(lab.sum()), "bins": len(lab), **dict(zip(shifts, acc))})
clock = pd.DataFrame(clock).set_index("session")
best = clock[shifts].idxmax(axis=1)
near = clock[[-3, 0, 3]].max(axis=1)
far = clock[[-60, 60]].max(axis=1)
print(f"{len(clock)} sessions. Shift of best accuracy: {best.value_counts().sort_index().to_dict()}")
print(f"best accuracy within 3 s of zero: median {near.median():.2f} (range {near.min():.2f} to {near.max():.2f}); "
      f"best at +-60 s: median {far.median():.2f}")
print(f"sessions where accuracy near zero exceeds that at +-60 s by less than 0.05: "
      f"{', '.join(clock.index[(near - far) < 0.05]) or 'none'}")
fig, ax = plt.subplots(figsize=(6, 3.2))
for _, r in clock.iterrows():
    ax.plot(shifts, r[shifts].to_numpy(), color="0.6", lw=0.7)
ax.plot(shifts, clock[shifts].median(), color="k", lw=2, label="median")
ax.set_xlabel("shift of the labels (s)"); ax.set_ylabel("balanced accuracy"); ax.axhline(0.5, ls="--", color="k", lw=0.8)
ax.legend(frameon=False); plt.tight_layout()
print(f"plotted accuracies range from {clock[shifts].min().min():.2f} to {clock[shifts].max().max():.2f}")

## Trial sets for the behaviour claims

The paper's Figure 1 uses 74 trials from 28 mice without listing them. The dandiset has female
annotations for 70 sessions; one of them (m764 natural-cycle day 1) is the same recording as
m764's primed session, so 69 distinct trials remain. Two sets are read: all of them, and the
receptive ones (any lordosis or wiggle). For the interbout interval, which needs only the male's
annotations, a third set adds every other distinct session outside the male-inhibition
experiment. A day of a 3-day series is not distinct when it was matched above to a separately
deposited session or has the same number of frames as one.

In [ ]:
dup_of = {}
for _, row in sessions_t[sessions_t.experiment.str.contains("3 days")].iterrows():
    others = sessions_t[(sessions_t.subject.str.lstrip("m") == row.subject.lstrip("m"))
                        & sessions_t.experiment.isin(["receptive", "ovariectomised, primed"])]
    for _, o in others.iterrows():
        matched = receptive_day.get(row.subject) == row.day and o.experiment == "receptive" and row.experiment.startswith("natural")
        if matched or o.n_frames == row.n_frames:
            dup_of[row.path] = o.path
print(f"{len(dup_of)} days of 3-day series duplicate a separately deposited session:")
for p, q in dup_of.items():
    print(f"  {p.split('_ecephys')[0]} = {q.split('_ecephys')[0]}")

distinct = sessions_t[~sessions_t.path.isin(dup_of)]
trial_sets = {
    "all annotated": distinct[distinct.female_annotated].path.tolist(),
    "receptive": distinct[distinct.female_annotated & distinct.receptive].path.tolist(),
    "all distinct, male annotation": distinct[distinct.experiment != "male inhibition"].path.tolist(),
}
for name, ps in trial_sets.items():
    subj = sessions_t.set_index("path").loc[ps, "subject"].str.lstrip("m")
    print(f"{name}: {len(ps)} trials from {subj.nunique()} mice")


def clipped(iv, t0, t1):
    return np.clip(iv, t0, t1)


def duration(s, labels, channel):
    t0, t1 = s.interaction()
    iv = clipped(vmhvl.intervals(s.behaviour, labels, channel), t0, t1)
    return float((iv[:, 1] - iv[:, 0]).sum())

## Figure 1b: accepting behaviour escalates over the interaction

> During mating the probability of accepting behaviours (lordosis, wiggle) increases gradually
> over the interaction, while resistance behaviours (dart, top up, kick, turn) first increase and
> then slowly decrease.

In [ ]:
BIN, SPAN = 20, 600
n_bins = SPAN // BIN


def group_curves(paths, labels):
    out = np.full((len(paths), n_bins), np.nan)
    for i, p in enumerate(paths):
        s = annotations[p]
        t0, t1 = s.interaction()
        frames = int(np.ceil(t1 * vmhvl.RATE)) + 1
        r = vmhvl.raster(s.behaviour, labels, frames, "Ch2")
        for k in range(n_bins):
            a, b = t0 + k * BIN, t0 + (k + 1) * BIN
            if b <= t1:
                out[i, k] = r[int(round(a * vmhvl.RATE)):int(round(b * vmhvl.RATE))].mean()
    return out


def judge_escalation(acc, res):
    counts = np.sum(~np.isnan(acc), 0)
    keep = counts >= 5
    ma, mr = np.nanmean(acc, 0)[keep], np.nanmean(res, 0)[keep]
    k = len(ma)
    rho, p = stats.spearmanr(np.arange(k), ma)
    third = k // 3
    first, last = ma[:third].mean(), ma[-third:].mean()
    run = np.convolve(mr, np.ones(3) / 3, "same")
    run[0], run[-1] = mr[:2].mean(), mr[-2:].mean()
    peak = int(np.argmax(run))
    accept_ok = rho > 0 and p < 0.05 and last > first
    resist_ok = 0 < peak < 2 * k / 3 and mr[-third:].mean() < run[peak]
    return dict(bins_kept=k, accept_spearman=round(rho, 2), accept_p=p, accept_first_third=round(first, 3),
                accept_last_third=round(last, 3), resist_peak_bin=peak, resist_peak=round(run[peak], 3),
                resist_first_bin=round(mr[0], 3), resist_last_third=round(mr[-third:].mean(), 3),
                accepting_escalates=accept_ok, resistance_rises_then_falls=resist_ok), ma, mr, keep


fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
rows = {}
for ax, name in zip(axes, ["all annotated", "receptive"]):
    ps = trial_sets[name]
    acc, res, app = (group_curves(ps, g) for g in (vmhvl.ACCEPT, vmhvl.RESIST, vmhvl.APPETITIVE))
    verdict, ma, mr, keep = judge_escalation(acc, res)
    rows[name] = verdict
    t = (np.arange(n_bins)[keep] + 0.5) * BIN
    for curve, label, color in ((acc, "accept", "C3"), (res, "resistance", "C0"), (app, "appetitive", "C2")):
        m = np.nanmean(curve, 0)[keep]
        se = np.nanstd(curve, 0)[keep] / np.sqrt(np.sum(~np.isnan(curve), 0)[keep])
        ax.plot(t, m, color=color, label=label); ax.fill_between(t, m - se, m + se, color=color, alpha=0.2)
    ax.set_title(f"{name} ({len(ps)} trials)", fontsize=10); ax.set_xlabel("time from male entry (s)")
    print(f"{name}: trials per bin from {np.sum(~np.isnan(acc), 0)[keep].min()} to {np.sum(~np.isnan(acc), 0)[keep].max()}; "
          f"plotted means range {min(np.nanmean(c, 0)[keep].min() for c in (acc, res, app)):.3f} to "
          f"{max(np.nanmean(c, 0)[keep].max() for c in (acc, res, app)):.3f}")
axes[0].set_ylabel("probability"); axes[0].legend(frameon=False); plt.tight_layout()
pd.DataFrame(rows).T

By mouse, for the receptive set: the Spearman correlation of each mouse's mean accepting curve
with time.

In [ ]:
ps = trial_sets["receptive"]
acc = group_curves(ps, vmhvl.ACCEPT)
subj = sessions_t.set_index("path").loc[ps, "subject"].str.lstrip("m").to_numpy()
by_mouse = []
for m in np.unique(subj):
    curve = np.nanmean(acc[subj == m], 0)
    ok = ~np.isnan(curve)
    by_mouse.append(dict(mouse=m, trials=int((subj == m).sum()), bins=int(ok.sum()),
                         spearman=round(stats.spearmanr(np.arange(ok.sum()), curve[ok])[0], 2)))
by_mouse = pd.DataFrame(by_mouse)
print(f"{(by_mouse.spearman > 0).sum()} of {len(by_mouse)} mice have a positive correlation")
by_mouse.set_index("mouse").T

**Result: reproduced.** Over the first 600 s after male entry, the mean probability of accepting
behaviour rises with time (Spearman 0.74, p = 3e-6, on all 69 annotated trials; 0.59, p = 6e-4, on
the 58 receptive trials), from 0.11 in the first third of the bins to 0.20 in the last. Resistance
peaks in the third bin and falls to 0.07 in the last third, under both sets. The rise is a
property of the pooled trials: within single mice the correlation is positive in only 12 of 23.

## Figure 1c: most female mating behaviour responds to the male

> Females spend 83.9% of their mating behaviour time in male-responsive behaviours (accepting,
> resistance, staying) and 16.1% in self-initiated ones (approach, sniff, check genital area).

In [ ]:
def behaviour_times(paths):
    rows = []
    for p in paths:
        s = annotations[p]
        t0, t1 = s.interaction()
        rows.append(dict(path=p, subject=sessions_t.set_index("path").subject[p].lstrip("m"),
                         interaction=t1 - t0,
                         responsive=duration(s, vmhvl.RESPONSIVE, "Ch2"),
                         female_self=duration(s, vmhvl.FEMALE_SELF, "Ch2"),
                         male_self=duration(s, vmhvl.MALE_SELF, "Ch1"),
                         accept=duration(s, vmhvl.ACCEPT, "Ch2")))
    return pd.DataFrame(rows)

times = {name: behaviour_times(trial_sets[name]) for name in ("all annotated", "receptive")}
rows = []
for name, t in times.items():
    frac = t.responsive / (t.responsive + t.female_self)
    rows.append({"trials": name, "n": len(t), "mean of per-trial fractions": round(frac.mean(), 3),
                 "SEM": round(frac.sem(), 3), "pooled fraction": round(t.responsive.sum() / (t.responsive + t.female_self).sum(), 3),
                 "range": f"{frac.min():.2f} to {frac.max():.2f}"})
print("expected 0.839, tolerance 0.084 (0.755 to 0.923)")
pd.DataFrame(rows).set_index("trials")

In [ ]:
t = times["receptive"]
t["fraction"] = t.responsive / (t.responsive + t.female_self)
print("Receptive trials, by mouse:")
t.groupby("subject").fraction.agg(["size", "mean", "min", "max"]).round(3).T

**Result: reproduced.** Male-responsive behaviours take 84.1% of female mating time on average
over the 69 annotated trials (83.5% pooled; 82.9% over receptive trials), against 83.9% in the
paper. Per mouse, the receptive-trial average runs from 65% to 94%.

## Figure 1d: the male drives the interaction

> Males spend 11.3 times more time in self-initiated mating behaviours (sniff, mount,
> intromission) than females do in theirs (approach, sniff, check genital area).

In [ ]:
rows = []
for name, t in times.items():
    ok = t.female_self > 0
    ratio = t.male_self[ok] / t.female_self[ok]
    rows.append({"trials": name, "n": len(t), "trials without female self-initiated time": int((~ok).sum()),
                 "mean of per-trial ratios": round(ratio.mean(), 2), "SEM": round(ratio.sem(), 2),
                 "ratio of summed times": round(t.male_self.sum() / t.female_self.sum(), 2),
                 "median per-trial ratio": round(ratio.median(), 2),
                 "range": f"{ratio.min():.1f} to {ratio.max():.1f}"})
print("expected 11.3, tolerance 1.13 (10.17 to 12.43)")
pd.DataFrame(rows).set_index("trials")

In [ ]:
t = times["receptive"]
t = t[t.female_self > 0].assign(ratio=lambda d: d.male_self / d.female_self)
print("Receptive trials, by mouse (per-trial ratio):")
t.groupby("subject").ratio.agg(["size", "mean", "min", "max"]).round(1).T

**Result: reproduced.** The mean over the 69 annotated trials of the per-trial ratio of male to
female self-initiated time is 11.75, within the tolerance of the paper's 11.3. The number depends
on the reading: the ratio of summed times is 7.5, the median per-trial ratio 9.2, and the mean
ratio over receptive trials 9.3, all outside the tolerance but all well above 1.

## Figure 1e: copulation bouts and interbout intervals

> Male copulation (mounting and intromission) comes in bouts separated by interbout intervals
> with a mean of 13.7 s (1,685 bouts and 1,611 intervals in 74 trials).

In [ ]:
def copulation(p, gap):
    s = annotations[p]
    return vmhvl.bouts(vmhvl.intervals(s.behaviour, vmhvl.COPULATION, "Ch1"), gap)

rows = []
for name, ps in trial_sets.items():
    for gap_name, gap in (("0.1 s", 0.1), ("1 s", 1.0), ("no merging", None)):
        b = [copulation(p, gap) for p in ps]
        ibi = np.concatenate([x[1:, 0] - x[:-1, 1] for x in b if len(x) > 1])
        dur = np.concatenate([x[:, 1] - x[:, 0] for x in b])
        rows.append({"trials": name, "merge gaps up to": gap_name, "trials n": len(ps),
                     "bouts": len(dur), "intervals": len(ibi), "mean interval (s)": round(ibi.mean(), 2),
                     "median interval (s)": round(np.median(ibi), 2), "mean bout (s)": round(dur.mean(), 2),
                     "intervals below 0": int((ibi < 0).sum())})
print("expected mean interval 13.7 s, tolerance 1.37 s (12.33 to 15.07)")
pd.DataFrame(rows).set_index(["trials", "merge gaps up to"])

In [ ]:
by_mouse = []
for p in trial_sets["receptive"]:
    x = copulation(p, 0.1)
    by_mouse.append(dict(subject=sessions_t.set_index("path").subject[p].lstrip("m"),
                         intervals=len(x) - 1 if len(x) else 0,
                         total=(x[1:, 0] - x[:-1, 1]).sum() if len(x) > 1 else 0.0))
by_mouse = pd.DataFrame(by_mouse).groupby("subject").sum()
by_mouse["mean interval (s)"] = (by_mouse.total / by_mouse.intervals).round(1)
print("Receptive trials, 0.1-s merging, by mouse:")
by_mouse[["intervals", "mean interval (s)"]].T

**Result: reproduced.** With mount and intromission annotations less than 0.1 s apart merged into
bouts, the mean interbout interval is 14.0 s over the 69 annotated trials, 14.9 s over receptive
trials and 14.6 s over all 88 distinct sessions, against 13.7 s in the paper. Merging gaps up to 1 s
lengthens it to 15.5 to 16.3 s, and not merging at all shortens it to 9.8 to 10.8 s; the paper's
figure matches the near-contiguous merging. The 69 trials give 1,540 bouts and 1,471 intervals.

## Figure 1g: females stay social during bouts and after them

> Females show social behaviours (accepting, resistance and appetitive) for 62% of the duration
> of each male copulation bout and continue them for 23% of each interbout interval.

In [ ]:
def social_fractions(ps, gap=0.1):
    in_bout, in_ibi, tb, sb, ti, si = [], [], 0, 0, 0, 0
    for p in ps:
        s = annotations[p]
        frames = int(np.ceil(s.behaviour.stop.max() * vmhvl.RATE)) + 2
        social = vmhvl.raster(s.behaviour, vmhvl.SOCIAL, frames, "Ch2")
        b = copulation(p, gap)
        for (a, z) in b:
            seg = social[int(round(a * vmhvl.RATE)):int(round(z * vmhvl.RATE)) + 1]
            in_bout.append(seg.mean()); tb += len(seg); sb += seg.sum()
        for (a, z) in zip(b[:-1, 1], b[1:, 0]):
            seg = social[int(round(a * vmhvl.RATE)) + 1:int(round(z * vmhvl.RATE))]
            if len(seg):
                in_ibi.append(seg.mean()); ti += len(seg); si += seg.sum()
    return np.array(in_bout), np.array(in_ibi), sb / tb, si / ti

rows = []
for name in ("all annotated", "receptive"):
    fb, fi, pb, pi = social_fractions(trial_sets[name])
    rows.append({"trials": name, "bouts": len(fb), "intervals": len(fi),
                 "bout, mean of fractions": round(fb.mean(), 3), "bout, pooled": round(pb, 3),
                 "interval, mean of fractions": round(fi.mean(), 3), "interval, pooled": round(pi, 3)})
print("expected 0.62 (0.558 to 0.682) in bouts and 0.23 (0.207 to 0.253) in intervals")
pd.DataFrame(rows).set_index("trials")

**Result: reproduced.** Over the 69 annotated trials, females show social behaviour for 64.9% of
each copulation bout and 23.0% of each interbout interval on average, against 62% and 23%. Both
depend on the reading: over receptive trials the interval value is 18.6%, and pooled over time, not averaged over
bouts, the values are 75% and 12%.

## Figure 2c: few cells are tuned to any one behaviour

> Only a small percentage of VMHvl alpha-cells (approximately 2 to 13%) are tuned to any one
> mating behaviour by choice probability; most show mixed selectivity.

In the 15 receptive sessions, each cell's z-scored trace is averaged in 1-s bins over the
interaction. A bin belongs to a behaviour when the behaviour covers at least half of it.

In [ ]:
rng_cp = np.random.default_rng(1097_02)
cp_behaviours = [(b, "Ch2") for b in vmhvl.FEMALE_MATING] + [(b, "Ch1") for b in vmhvl.MALE_SELF]
cp_rows = []
for p in receptive_paths:
    s = data[p]
    t0, t1 = s.interaction()
    i0, i1 = int(round(t0 * vmhvl.RATE)), min(int(round(t1 * vmhvl.RATE)), s.n_frames)
    y = vmhvl.bin_frames(vmhvl.zscore(s.traces)[i0:i1], 10)
    for b, ch in cp_behaviours:
        lab = vmhvl.bin_frames(vmhvl.raster(s.behaviour, [b], s.n_frames, ch)[i0:i1].astype(float), 10) >= 0.5
        if lab.sum() < 10 or (~lab).sum() < 10:
            continue
        cp, mu, sd = vmhvl.choice_probability(y, lab, rng_cp)
        tuned = ((cp > 0.7) | (cp < 0.3)) & (np.abs(cp - mu) > 2 * sd)
        cp_rows.append(dict(subject=s.subject, behaviour=b, bins=int(lab.sum()), cells=s.n_cells,
                            tuned=int(tuned.sum()), tuned_up=int((tuned & (cp > 0.5)).sum()),
                            beyond_2sd_only=int((np.abs(cp - mu) > 2 * sd).sum())))
cp_rows = pd.DataFrame(cp_rows)
mice = cp_rows.groupby("behaviour").subject.nunique()
analysed = mice[mice >= 5].index
cp_table = cp_rows[cp_rows.behaviour.isin(analysed)].groupby("behaviour").agg(
    mice=("subject", "nunique"), cells=("cells", "sum"), tuned=("tuned", "sum"), tuned_up=("tuned_up", "sum"),
    beyond_2sd=("beyond_2sd_only", "sum"))
cp_table["tuned %"] = (100 * cp_table.tuned / cp_table.cells).round(1)
cp_table["beyond 2 SD of shuffles, any CP %"] = (100 * cp_table.beyond_2sd / cp_table.cells).round(1)
print(f"behaviours with at least 10 bins in at least 5 mice: {len(analysed)}; left out: "
      f"{', '.join(f'{b} ({n} mice)' for b, n in mice[mice < 5].items()) or 'none'}")
print(f"median tuned % over behaviours: {cp_table['tuned %'].median():.1f}; range {cp_table['tuned %'].min():.1f} to {cp_table['tuned %'].max():.1f}")
cp_table.sort_values("tuned %")

In [ ]:
per_mouse = cp_rows[cp_rows.behaviour.isin(analysed)].assign(pct=lambda d: 100 * d.tuned / d.cells)
print("Tuned % by mouse (rows) and behaviour (columns); blank where the mouse had fewer than 10 bins:")
per_mouse.pivot(index="subject", columns="behaviour", values="pct").round(0)

**Result: not reproduced.** With the procedure set out in the criterion, 6.4% (approach) to 35.3%
(wiggle) of cells are tuned to a behaviour, with a median over the 12 behaviours of 20.8%, against
about 2 to 13% in the paper. Nine of the 12 behaviours exceed 15%. The comparison set is a choice
here: the Methods compare a behaviour with "another", and Figure 2c does not say which. Comparing
with every other bin of the interaction, as here, may tune more cells than comparing two specific
behaviours. The shuffle criterion alone (beyond 2 SD of shuffled labels) passes 18% to 75% of cells;
the 0.7 and 0.3 cut-offs remove most of these.

## Figure 2e and 2f: behaviour explains little of a cell's activity; other cells explain more

> A linear model of each neuron's activity from behaviour explains little of its variance, with
> mean cross-validated R2 of 0.08 from female behaviours, 0.14 from male behaviours and 0.15
> from both.

> Adding the activity of the other simultaneously imaged neurons as regressors raises the mean
> cross-validated R2 to 0.46.

Ridge regression of each z-scored trace, over the interaction, on the behaviour rasters
delayed by 0 to 10 s in 1-s steps; tenfold contiguous cross-validation.

In [ ]:
lags = np.arange(0, 101, 10)
rng_glm = np.random.default_rng(1097_03)
glm, glm_random = [], []
for p in receptive_paths:
    s = data[p]
    t0, t1 = s.interaction()
    i0, i1 = int(round(t0 * vmhvl.RATE)), min(int(round(t1 * vmhvl.RATE)), s.n_frames)
    Y = vmhvl.zscore(s.traces)[i0:i1]
    fem = np.column_stack([vmhvl.raster(s.behaviour, [b], s.n_frames, "Ch2")[i0:i1] for b in vmhvl.FEMALE_MATING])
    mal = np.column_stack([vmhvl.raster(s.behaviour, [b], s.n_frames, "Ch1")[i0:i1] for b in vmhvl.MALE_SELF])
    Xf, Xm = vmhvl.lagged(fem.astype(float), lags), vmhvl.lagged(mal.astype(float), lags)
    Xb = np.hstack([Xf, Xm])
    r2 = {"female": vmhvl.cv_ridge(Xf, Y), "male": vmhvl.cv_ridge(Xm, Y), "both": vmhvl.cv_ridge(Xb, Y),
          "coupling": vmhvl.cv_ridge(np.hstack([Xb, Y]), Y, exclude=Xb.shape[1] + np.arange(Y.shape[1]))}
    r2r = {"female": vmhvl.cv_ridge(Xf, Y, rng=rng_glm), "male": vmhvl.cv_ridge(Xm, Y, rng=rng_glm),
           "both": vmhvl.cv_ridge(Xb, Y, rng=rng_glm),
           "coupling": vmhvl.cv_ridge(np.hstack([Xb, Y]), Y, exclude=Xb.shape[1] + np.arange(Y.shape[1]), rng=rng_glm)}
    for i in range(Y.shape[1]):
        glm.append(dict(subject=s.subject, cell=i, **{k: v[i] for k, v in r2.items()}))
        glm_random.append(dict(subject=s.subject, cell=i, **{k: v[i] for k, v in r2r.items()}))
glm, glm_random = pd.DataFrame(glm), pd.DataFrame(glm_random)
models = ["female", "male", "both", "coupling"]
expected = {"female": 0.08, "male": 0.14, "both": 0.15, "coupling": 0.46}
summary = pd.DataFrame({
    "expected": expected,
    "mean over cells": glm[models].mean(),
    "mean of mouse means": glm.groupby("subject")[models].mean().mean(),
    "median over cells": glm[models].median(),
    "cells below 0": (glm[models] < 0).sum(),
    "variant, random folds, mean over cells": glm_random[models].mean(),
    "variant, random folds, mean of mouse means": glm_random.groupby("subject")[models].mean().mean(),
}).round(3)
print(f"{len(glm)} cells from {glm.subject.nunique()} mice; cvR2 range over cells: "
      + "; ".join(f"{m} {glm[m].min():.2f} to {glm[m].max():.2f}" for m in models))
summary

In [ ]:
print("Mean cvR2 by mouse:")
glm.groupby("subject")[models].mean().round(3).T

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
for i, m in enumerate(models):
    ax.violinplot(np.clip(glm[m], -0.2, 1), positions=[i], showmedians=True)
ax.set_xticks(range(4), models); ax.set_ylabel("cvR2 (clipped at -0.2)"); plt.tight_layout()
print(f"values below -0.2 clipped: " + ", ".join(f"{m} {(glm[m] < -0.2).sum()}" for m in models))

**Result: not reproduced.** With contiguous folds, the mean cross-validated R2 is -0.078 from female
behaviours, -0.011 from male behaviours and -0.048 from both, against 0.08, 0.14 and 0.15; means of
mouse means are -0.050, 0.010 and -0.012. Most cells (829 to 1,018 of 1,335) score below zero,
because slow changes in a trace are not predicted in a held-out block of time. The paper's point,
that behaviour explains little of the activity, holds, but the numbers and the ordering do not
(both behaviours together do worse than male behaviours alone). With frames assigned to folds at
random, a variant the criterion did not name, the means are 0.234, 0.207 and 0.323, above the
paper's values.

**Result: reproduced.** Adding the instantaneous activity of the other cells raises the mean
cross-validated R2 to 0.450 over cells (0.434 over mouse means), against 0.46 in the paper; 75 of
1,335 cells remain below zero. With random folds the value is 0.815.

## Figure 2i: many cells are persistent

> About 30% of cells have an autocorrelation half-width (ACHW) longer than 25 s, and the mean
> ACHW is 20 s, longer than the mean interbout interval.

In [ ]:
achw_rows = []
for p in receptive_paths:
    s = data[p]
    t0, t1 = s.interaction()
    i0, i1 = int(round(t0 * vmhvl.RATE)), min(int(round(t1 * vmhvl.RATE)), s.n_frames)
    for window, (a, b) in (("whole recording", (0, s.n_frames)), ("interaction", (i0, i1))):
        for smoothed in (True, False):
            y = vmhvl.prepare(s.traces[a:b], smoothed=smoothed)
            for first in (False, True):
                h = vmhvl.achw(y, first_crossing=first)
                achw_rows.append(dict(subject=s.subject, window=window, smoothed=smoothed,
                                      rule="first crossing" if first else "closest to 0.5 (code)",
                                      cells=len(h), above_25=(h > 25).sum(), sum=h.sum(), at_cap=(h >= 50).sum()))
achw_rows = pd.DataFrame(achw_rows)
achw_table = achw_rows.groupby(["window", "smoothed", "rule"])[["cells", "above_25", "sum", "at_cap"]].sum()
achw_table["fraction above 25 s"] = (achw_table.above_25 / achw_table.cells).round(3)
achw_table["mean ACHW (s)"] = (achw_table["sum"] / achw_table.cells).round(1)
print("expected fraction 0.30 (0.27 to 0.33) and mean 20 s (18 to 22); ACHW is capped at 50 s (500 lags)")
achw_table[["cells", "fraction above 25 s", "mean ACHW (s)", "at_cap"]]

In [ ]:
main = achw_rows[(achw_rows.window == "whole recording") & achw_rows.smoothed & (achw_rows.rule == "closest to 0.5 (code)")]
print("Whole recording, smoothed, code's rule, by mouse:")
main.assign(fraction=lambda d: (d.above_25 / d.cells).round(2), mean=lambda d: (d["sum"] / d.cells).round(1)
            ).set_index("subject")[["cells", "fraction", "mean"]].T

**Result: not reproduced.** Over the whole recording, smoothed as in the authors' code, 20.0% of
cells have an ACHW above 25 s and the mean ACHW is 15.5 s, against 30% and 20 s. Every other reading
gives the same or less: the first-crossing rule 19.9% and 15.5 s, the others 10 to 17%
and 9.9 to 13.1 s. The mouse-level fractions range from 0 to 0.48, and four
mice (m540, m551, m930, m940) reach 0.28 or more; Figure 2i's lower panel has n = 4 mice, so the
paper's 30% may come from a subset, which the paper does not identify. The mean ACHW of 15.5 s is
still longer than the mean interbout interval of 14.0 s found above.

## Extended Data Figure 6e and Figure 5h: the receptive-day dimension on unreceptive days

> Projected onto the integration dimension found on the receptive day, activity on unreceptive
> days does not ramp and stays low during male mounting (normalised value 0.09 on unreceptive
> against 0.69 on receptive days, n = 4 mice).

The authors' rSLDS weights are deposited for two mice: in `sub-m541/sub-m541_behavior.nwb`,
whose cells are those of m541's natural-cycle days, and in `sub-m542/sub-m542_behavior.nwb`,
whose cells are those of mouse 543 (shown in the data check). The receptive days are m541 day 3
and 543 day 1, matched above.

In [ ]:
cycle_mice = {"m541": ("m541", "sub-m541_ses-WT_3d_mating_d{}_ecephys+behavior.nwb"),
              "543": ("m542", "sub-543_ses-WT_3d_mating_d{}_ecephys+behavior.nwb")}
proj = {}
for mouse, (wfile, pattern) in cycle_mice.items():
    rday = int(receptive_day[mouse])
    days = {d: data[pattern.format(d)] for d in (1, 2, 3)}
    W = weights[wfile]["weights"]
    rec = days[rday]
    t_entry = int(round(rec.interaction()[0] * vmhvl.RATE))
    capped = [vmhvl.achw((rec.traces @ W[:, k])[:, None])[0] for k in range(W.shape[1])]
    # The 500-lag cap of the authors' ACHW can tie; the factor is chosen with lags up to 300 s.
    halfwidths = [vmhvl.achw((rec.traces @ W[:, k])[:, None], max_lag=3000)[0] for k in range(W.shape[1])]
    chosen = int(np.argmax(halfwidths))
    print(f"mouse {mouse} (weights from sub-{wfile}): ACHW of the receptive-day projection, capped at 50 s, "
          f"factor 1 {capped[0]:.1f} s and factor 2 {capped[1]:.1f} s; up to 300 s, {halfwidths[0]:.1f} s and "
          f"{halfwidths[1]:.1f} s; integration factor {chosen + 1}")
    for k in (chosen, 1 - chosen):
        x_rec = rec.traces @ W[:, k]
        sign = 1 if x_rec[t_entry:].mean() > x_rec[:t_entry].mean() else -1
        w = sign * W[:, k]
        xs = {d: s.traces @ w for d, s in days.items()}
        lo, hi = min(x.min() for x in xs.values()), max(x.max() for x in xs.values())
        rlo, rhi = xs[rday].min(), xs[rday].max()
        rows = []
        for d, s in days.items():
            mount = vmhvl.raster(s.behaviour, ["mount"], s.n_frames, "Ch1")
            t0, t1 = s.interaction()
            e = int(round(t0 * vmhvl.RATE))
            end = min(int(round(t1 * vmhvl.RATE)), s.n_frames)
            x = xs[d]
            rows.append({"day": d, "receptive": d == rday, "mount frames": int(mount.sum()),
                         "intromission bouts": int((s.behaviour.behavior == "intromission").sum()),
                         "mounting, min-max over 3 days": round(((x[mount] - lo) / (hi - lo)).mean(), 3),
                         "mounting, receptive-day range": round(((x[mount] - rlo) / (rhi - rlo)).mean(), 3),
                         "ramp: last 100 s minus 60 s before entry (SD of receptive day)":
                             round((x[end - 1000:end].mean() - x[e - 600:e].mean()) / xs[rday].std(), 2)})
        table = pd.DataFrame(rows).set_index("day")
        if k == chosen:
            proj[mouse] = dict(factor=k + 1, sign=sign, xs=xs, rday=rday, lo=lo, hi=hi, table=table)
            print(f"  factor {k + 1} (the integration factor), sign {sign:+d}:")
        else:
            print(f"  variant, factor {k + 1}, sign {sign:+d}:")
        print(table.to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, (mouse, pr) in zip(axes, proj.items()):
    for d, x in pr["xs"].items():
        s = data[cycle_mice[mouse][1].format(d)]
        e = int(round(s.interaction()[0] * vmhvl.RATE))
        t = (np.arange(len(x)) - e) / vmhvl.RATE
        ax.plot(t, (x - pr["lo"]) / (pr["hi"] - pr["lo"]), lw=0.8,
                label=f"day {d}" + (" (receptive)" if d == pr["rday"] else ""))
    ax.set_title(f"mouse {mouse}", fontsize=10); ax.set_xlabel("time from male entry (s)"); ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("integration factor, min-max over 3 days"); plt.tight_layout()
print("all plotted values lie in 0 to 1 by construction")

**Result: partially reproduced.** In m541, on the integration factor (factor 2, chosen by its
receptive-day ACHW of 105 s against 57 s), the normalised value during mounting is 0.62 on the
receptive day against 0.23 and 0.19 on the unreceptive days, close to the paper's 0.69 against
0.09. In mouse 543 (factor 1, ACHW 129 s against 32 s) it is 0.61 on the receptive day against 0.64
and 0.63, so not higher; on that mouse's other factor it is higher (0.66 against 0.54 and 0.51). The
ramp part of the claim holds in both mice: from before male entry to the last 100 s of the
interaction the projection rises by 2.5 and 2.1 receptive-day SDs on the receptive day, against
0.0 to 0.5 and -0.4 to 0.2 on the unreceptive days. On factor 1, m541's receptive day is also
higher, with a smaller gap (0.62 against 0.48 and 0.49).

## Extended Data Figure 7: single-cell persistence across the cycle

> Cells weighted on the integration dimension have shorter ACHW on unreceptive than on receptive
> days (16.1 +- 0.8 s against 25.2 +- 1.5 s, P < 0.001), while cells that do not contribute to
> it do not change.

In [ ]:
ed7 = []
for mouse, (wfile, pattern) in cycle_mice.items():
    pr = proj[mouse]
    w = weights[wfile]["weights"][:, pr["factor"] - 1]
    weighted = np.abs(w) > np.std(w, ddof=1)
    h = {d: vmhvl.achw(vmhvl.prepare(data[pattern.format(d)].traces)) for d in (1, 2, 3)}
    unrec = [d for d in (1, 2, 3) if d != pr["rday"]]
    for group, mask in (("weighted", weighted), ("other", ~weighted)):
        hr = h[pr["rday"]][mask]
        comparisons = {"unreceptive pooled": np.concatenate([h[d][mask] for d in unrec])}
        comparisons.update({f"unreceptive day {d}": h[d][mask] for d in unrec})
        for name, hu in comparisons.items():
            ks = stats.ks_2samp(hr, hu)
            ed7.append({"mouse": mouse, "cells": group, "n": int(mask.sum()), "against": name,
                        "receptive mean (s)": round(hr.mean(), 1), "receptive SEM": round(stats.sem(hr), 1),
                        "unreceptive mean (s)": round(hu.mean(), 1), "unreceptive SEM": round(stats.sem(hu), 1),
                        "KS statistic": round(ks.statistic, 2), "KS p": ks.pvalue})
ed7 = pd.DataFrame(ed7).set_index(["mouse", "cells", "against"])
print("expected: weighted cells 25.2 s on receptive against 16.1 s on unreceptive days; other cells unchanged")
ed7

In [ ]:
print("Variants: the one-SD threshold of the authors' code replaced by the top 10% and the top 25% of |weight|,\n"
      "and the one-SD threshold on the other factor; unreceptive days pooled.")
rows = []
for mouse, (wfile, pattern) in cycle_mice.items():
    pr = proj[mouse]
    W = np.abs(weights[wfile]["weights"])
    h = {d: vmhvl.achw(vmhvl.prepare(data[pattern.format(d)].traces)) for d in (1, 2, 3)}
    unrec = [d for d in (1, 2, 3) if d != pr["rday"]]
    k, other = pr["factor"] - 1, 2 - pr["factor"]
    selections = {f"factor {k + 1}, top 10%": W[:, k] > np.quantile(W[:, k], 0.90),
                  f"factor {k + 1}, top 25%": W[:, k] > np.quantile(W[:, k], 0.75),
                  f"factor {other + 1}, above 1 SD": W[:, other] > np.std(weights[wfile]["weights"][:, other], ddof=1)}
    for name, mask in selections.items():
        for group, m in (("weighted", mask), ("other", ~mask)):
            hr, hu = h[pr["rday"]][m], np.concatenate([h[d][m] for d in unrec])
            rows.append({"mouse": mouse, "selection": name, "cells": group, "n": int(m.sum()),
                         "receptive mean": round(hr.mean(), 1), "unreceptive mean": round(hu.mean(), 1),
                         "KS p": stats.ks_2samp(hr, hu).pvalue})
pd.DataFrame(rows).set_index(["mouse", "selection", "cells"])

**Result: partially reproduced.** In both mice, cells weighted on the integration factor have longer
ACHWs on the receptive day than on the unreceptive days: 28.4 against 17.2 s in m541 (KS p = 2e-5)
and 26.7 against 9.1 s in 543 (KS p = 2e-4), against 25.2 and 16.1 s in the paper. The other cells
do not differ in m541 (14.9 against 12.1 s, KS p = 0.23) but do in 543 (12.8 against 10.8 s, KS
p = 0.005). With the top 25% of weights in place of the one-SD threshold the same pattern holds;
with the top 10%, the weighted cells of 543 (8 cells) no longer differ (p = 0.13).

## Extended Data Figure 10c: the population mean does not track receptivity

> In the same trials, the area under the curve of the population mean of all neurons does not
> correlate with the percentage of time spent in accepting behaviours.

In [ ]:
ed10 = []
for p in oestrogen_paths:
    s = data[p]
    t0, t1 = s.interaction()
    e, end = int(round(t0 * vmhvl.RATE)), min(int(round(t1 * vmhvl.RATE)), s.n_frames)
    m = s.traces.mean(1)
    base = m[max(0, e - 600):e].mean()
    contact = s.behaviour[(s.behaviour.channel == "Ch1") & s.behaviour.behavior.isin(vmhvl.MALE_SELF)].start.min()
    c = int(round(contact * vmhvl.RATE))
    ed10.append({"session": p.split("_ecephys")[0], "mouse": s.subject,
                 "accepting %": 100 * duration(s, vmhvl.ACCEPT, "Ch2") / (t1 - t0),
                 "AUC entry to end, baseline-subtracted": (m[e:end] - base).sum() / vmhvl.RATE,
                 "AUC entry to end, raw": m[e:end].sum() / vmhvl.RATE,
                 "AUC 200 s from first contact, baseline-subtracted": (m[c:c + 2000] - base).sum() / vmhvl.RATE})
ed10 = pd.DataFrame(ed10)
rows = []
for col in ed10.columns[3:]:
    r = stats.linregress(ed10["accepting %"], ed10[col])
    rows.append({"reading": col, "trials": len(ed10), "r": round(r.rvalue, 3), "r2": round(r.rvalue ** 2, 3),
                 "p": r.pvalue, "slope": r.slope})
print(f"{len(ed10)} trials from {ed10.mouse.nunique()} mice; accepting % from {ed10['accepting %'].min():.1f} to "
      f"{ed10['accepting %'].max():.1f}; {int((ed10['accepting %'] == 0).sum())} trials without accepting behaviour")
pd.DataFrame(rows).set_index("reading")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.4))
col = "AUC entry to end, baseline-subtracted"
for i, (mouse, g) in enumerate(ed10.groupby("mouse")):
    ax.scatter(g["accepting %"], g[col], s=18, label=mouse, color=f"C{i}")
ax.set_xlabel("accepting behaviour (% of interaction)"); ax.set_ylabel("population mean AUC (z x s)")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()
print(f"plotted AUC range {ed10[col].min():.0f} to {ed10[col].max():.0f}")
print("By mouse, Spearman correlation of accepting % with this AUC:")
print(ed10.groupby("mouse").apply(lambda g: round(stats.spearmanr(g["accepting %"], g[col])[0], 2)).to_dict())

**Result: reproduced.** Over the 42 trials, the area under the population mean does not
correlate with the percentage of time in accepting behaviour under any of the three readings
(r2 = 0.015, 0.003 and 0.027; p = 0.45, 0.73 and 0.30). Within mice the Spearman correlations run
from -0.67 to 0.52.

## Claims that cannot be tested here

The cell below records what the dandiset holds for the blocked claims, so that the reasons can
be checked.

In [ ]:
print("Male-inhibition sessions (Figure 4c,d) and their inhibition epochs:")
for p in sessions_t[sessions_t.experiment == "male inhibition"].path:
    b = annotations[p].behaviour
    inh = b[b.behavior.isin(["laser_on", "mating_inhibition"])]
    print(f"  {p.split('_ecephys')[0]}: " + "; ".join(f"{r.behavior} {r.start:.0f} to {r.stop:.0f} s ({r.stop - r.start:.0f} s)" for r in inh.itertuples()))
print()
ej = b_all[b_all.behavior == "ejaculation"].path.unique()
print(f"Sessions with an annotated ejaculation (Figure 4g,h): {len(ej)}: {', '.join(p.split('_ecephys')[0] for p in ej)}")
print()
print("Labels that could mark a pencil-cup period (Extended Data Figure 3a-c):",
      [l for l in b_all.behavior.unique() if "cup" in l.lower() or "pencil" in l.lower()] or "none")
print("Labels that could mark a 10-s optogenetic inhibition of VMHvl (Figure 3h-k):",
      [l for l in b_all.behavior.unique() if "led" in l.lower() or "nvoke" in l.lower()] or "none")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome | Note |
|---|---|---|
| `fig1b-accepting-escalates` | Reproduced | Accepting rises (Spearman 0.74); resistance peaks early |
| `fig1c-responsive-dominates` | Reproduced | 84.1% against 83.9% |
| `fig1d-male-drives-mating` | Reproduced | 11.75 against 11.3, on one of six readings |
| `fig1e-interbout-interval` | Reproduced | 14.0 s against 13.7 s |
| `fig1g-social-in-bouts-and-intervals` | Reproduced | 64.9% and 23.0% against 62% and 23% |
| `fig2c-few-tuned-cells` | Not reproduced | 6 to 35% tuned, median 21% |
| `fig2e-glm-behaviour-explains-little` | Not reproduced | cvR2 below zero with contiguous folds |
| `fig2f-glm-coupling` | Reproduced | 0.450 against 0.46 |
| `fig2i-persistent-cells` | Not reproduced | 20% above 25 s, mean 15.5 s |
| `ed3b-pencil-cup` | Blocked: metadata insufficient | The pencil-cup period is not marked |
| `fig3a-rslds-fit`, `fig3c-integration-dimension`, `ed3s-integrator-model`, `ed4b-single-cell-ramps`, `fig4b-persistence-during-intervals`, `fig4d-persistence-male-inhibition`, `fig4h-peak-before-ejaculation`, `fig5c-attractor-receptive-days`, `ed9j-ovx-hormone-priming`, `fig5k-receptivity-correlation` | Blocked: other | The rSLDS cannot be built or fit within the registry's environment and time limit |
| `fig3k-perturbation-relaxation` | Blocked: data missing | The VMHvl inhibition sessions are not in the dandiset |
| `ed6e-projection-unreceptive-days` | Partially reproduced | Holds in m541, not in 543 |
| `ed7-achw-across-cycle` | Partially reproduced | Weighted cells hold in both mice; other cells change in 543 |
| `ed10c-population-mean-uncorrelated` | Reproduced | r2 at most 0.03 |

### How this differs from the paper's analysis

- No rSLDS model is fitted, so the paper's central claims about the line attractor are not tested
  directly. The two tested claims about it use the authors' deposited weights for two mice where
  the paper has four, and choose the integration factor by its autocorrelation, since the files do
  not say which factor it is.
- The receptive day of a natural-cycle series is the day matching the mouse's separately deposited
  receptive session, since the files do not mark it and the female's behaviour is not annotated
  on those days.
- Behaviour claims use 69 distinct annotated trials from 23 mice, where the paper has 74 from 28.
- The GLM uses lags in 1-s steps, contiguous folds and a penalty chosen on held-out training data;
  choice probability compares a behaviour with every other bin of the interaction; the ACHW
  follows the authors' code. None of these is fixed by the Methods.

### What this attempt found about the dandiset

- It holds the imaging and annotations behind Figures 1, 2 and 5, and the male-inhibition sessions
  of Figure 4, but no rSLDS fits or latent factors beyond the per-cell weights of two mice, though
  its description says it contains latent factors. The VMHvl inhibition sessions of Figure 3h-k
  are not in it.
- The per-mouse file labelled m542 holds the neural data of mouse 543 (72 cells; its traces match
  543's receptive day cell by cell), and mouse 543 is labelled `543` in some files and `m543` in
  another.
- The same recording appears under two experiments twice: m764's natural-cycle day 1 is its
  primed ovariectomy session, and m776's primed session has the same traces as its
  male-inhibition session, with different annotations.
- Which natural-cycle days were receptive is not recorded, and the female's behaviour is not
  annotated on most of them. The paper's subsets (4 of 7 natural-cycle mice, 4 of 8
  male-inhibition mice, 50 oestrogen trials where 42 are deposited) are not identified.
- The imaging metadata describe a two-photon microscope at 920 nm where the paper used a one-photon
  miniscope, the raw traces are labelled lumens, the z-scored traces are not z-scored per file,
  the image masks are 1 by 1 pixel, and experimenter, lab, institution and session dates are
  placeholders.